## 1. Check GPU
First, we check if a GPU is available. If you see an error or no GPU, go to **Runtime -> Change runtime type -> T4 GPU -> Save**.

In [ ]:
!nvidia-smi
import torch
if not torch.cuda.is_available():
    print("\n!!! NO GPU FOUND !!!")
    print("Go to Runtime menu -> Change runtime type -> T4 GPU -> Save.")
else:
    print(f"\nGPU attached: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive
This will connect your Google Drive so we can access `margdrishti_colab.zip` and save training results directly to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Extract the Project
Copy the zip file from Drive to Colab and unzip it. This creates a folder `/content/MargDrishti`.

In [ ]:
!cp /content/drive/MyDrive/MargDrishti_Colab/margdrishti_colab.zip /content/
!unzip -q -o /content/margdrishti_colab.zip -d /content/MargDrishti
import os
os.chdir('/content/MargDrishti')
print(f"Current directory: {os.getcwd()}")

## 4. Install Dependencies
Install the required packages. Colab already has PyTorch installed, so we skip it to save time.

In [ ]:
!grep -v "torch" requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
print(f"PyTorch version: {torch.__version__}")

## 5. Setup Experiments Folder on Drive
We create a symbolic link so that all training checkpoints and results are saved directly to your Google Drive (`MyDrive/MargDrishti_Colab/experiments`). If Colab disconnects, your progress is safe!

In [ ]:
drive_experiments = "/content/drive/MyDrive/MargDrishti_Colab/experiments"
import os
os.makedirs(drive_experiments, exist_ok=True)
if os.path.exists("experiments"):
    if not os.path.islink("experiments"):
        import shutil
        shutil.rmtree("experiments")
        os.symlink(drive_experiments, "experiments")
else:
    os.symlink(drive_experiments, "experiments")
print("Experiments folder linked to Google Drive.")

## 6. Run Training (Ablation)
This runs all five custom CNN steps. If it disconnects halfway, just run the cells above again, and then this one. It will skip completed runs and resume the unfinished one from `last.pt` automatically.

In [ ]:
# RUNS specifies which models to train. Leave empty or None to run all.
# e.g., RUNS = "v3,v5"
RUNS = None

cmd = "python -m scripts.run_ablation"
if RUNS:
    cmd += f" --only {RUNS}"
    
!{cmd}

## 7. Download Results
This zips the `experiments` folder and triggers a download to your laptop. Save this file to the root of your MargDrishti project!

In [ ]:
!zip -r -q /content/drive/MyDrive/MargDrishti_Colab/margdrishti_results_custom.zip experiments/
from google.colab import files
files.download('/content/drive/MyDrive/MargDrishti_Colab/margdrishti_results_custom.zip')

## 8. Run ResNet50 Stage 1 (Frozen Backbone)
This trains only the new classification head while keeping the pretrained ResNet50 features frozen. The first run will download the ResNet50 weights from torchvision (about 100 MB). If it disconnects, just re-run this cell.

In [ ]:
!python -m scripts.train --config configs/experiments/resnet50_frozen.yaml --run-name resnet50_frozen

## 9. Run ResNet50 Stage 2 (Fine-Tuning)
This unfreezes all layers and trains the whole model with a very small learning rate. It requires the Stage 1 checkpoint to exist. If it disconnects, just re-run this cell and it will resume from last.pt.

In [ ]:
import os
if not os.path.exists("experiments/resnet50_frozen/best.pt"):
    print("Error: Stage 1 (frozen) best.pt not found. Please run Stage 1 completely before fine-tuning.")
else:
    !python -m scripts.train --config configs/experiments/resnet50_finetune.yaml --run-name resnet50_finetune

## 10. Download ResNet50 Results
This zips the two ResNet50 folders and downloads them to your laptop. Save this file to the root of your MargDrishti project!

In [ ]:
!zip -r -q /content/drive/MyDrive/MargDrishti_Colab/margdrishti_results_resnet.zip experiments/resnet50_frozen experiments/resnet50_finetune
from google.colab import files
files.download("/content/drive/MyDrive/MargDrishti_Colab/margdrishti_results_resnet.zip")

## If something goes wrong...
- **Out of Memory?** In `configs/base.yaml` (or the specific experiment yaml), reduce the `batch_size` (e.g., from 64 to 32).
- **See GPU Memory:** Run `!nvidia-smi` in a new cell to check how much memory is used.
- **Stuck or Frozen?** Go to **Runtime -> Restart runtime** and start from the beginning. It will resume where it left off.

## 4. Bonus: YOLO Detection Training
Train the YOLO detector. Ensure you packed the detection data using --detection.

In [ ]:
!pip install ultralytics
!python -m scripts.train_detector

import shutil
shutil.make_archive('margdrishti_results_yolo', 'zip', 'experiments', 'yolo_nano')
from google.colab import files
files.download('margdrishti_results_yolo.zip')